In [9]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# LSTM Model Conceptualization

The following framework is followed for the LSTM model of the subway:

- Input Sequence: At each point in time (counted with 30 minute intervals), the following features are passed on:
    1. each station is placed as a categorical variable of whether or not a delay was recorded (this is as min delays is not a reliable measure for subway data)
    2. the day of the week (integer valued between 0-6)
    3. the datetime as an integer
- Training: Data between 2016~2024-01-01 or 2020~2024-01-01 (the effectiveness of both should be tested and compared)
- Output Sequence: Given a certain sequence segment predict the next step in the sequence. We can utilize unseen data to do the testing for this

# Preprocess the Data

## Load the Data

In [10]:
file_name = "../data/cleaned_subway_data.csv"
out_data_name = "../data/lstm_data.csv"

sub_data = pd.read_csv(file_name)

## Convert data to sequential data of desired form

In [11]:
sub_data["10_min_time"] = pd.to_datetime(sub_data["Datetime"])
sub_data["10_min_time"] = sub_data["10_min_time"].dt.floor("10T")
fixed_sub_data = sub_data.groupby(["10_min_time", "Day","Station Name"])["Min Delay"].count().reset_index()
fixed_sub_data = pd.get_dummies(fixed_sub_data, columns=["Station Name"], drop_first=True)
grouping_col = fixed_sub_data["10_min_time"]

fixed_sub_data = fixed_sub_data.groupby(grouping_col).max().reset_index()

day_conversion = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

fixed_sub_data['Day'] = fixed_sub_data['Day'].map(day_conversion)

/var/folders/xt/5my4_t657l5dvcsk9ybkb_ww0000gn/T/ipykernel_64317/1612561953.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  sub_data["10_min_time"] = sub_data["10_min_time"].dt.floor("10T")


In [12]:
date_range = pd.date_range(start="2016-01-01", end="2025-01-01", freq="30T")
dummy = pd.DataFrame({'10_min_time': date_range})

/var/folders/xt/5my4_t657l5dvcsk9ybkb_ww0000gn/T/ipykernel_64317/3084945216.py:1: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(start="2016-01-01", end="2025-01-01", freq="30T")


In [29]:
lstm_data = dummy.merge(fixed_sub_data, on='10_min_time', how='outer')
station_cols = [col for col in lstm_data.columns if col.startswith('Station Name')]
lstm_data[station_cols] = lstm_data[station_cols].fillna(False)

lstm_data['10_min_time'] = pd.to_datetime(lstm_data['10_min_time'])
lstm_data['Day'] = lstm_data['10_min_time'].dt.dayofweek

lstm_data['Min Delay'] = lstm_data['Min Delay'].fillna(0)

display(lstm_data.head())

/var/folders/xt/5my4_t657l5dvcsk9ybkb_ww0000gn/T/ipykernel_64317/483236953.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lstm_data[station_cols] = lstm_data[station_cols].fillna(False)


,10_min_time,Day,Min Delay,Station Name_BAY STATION,Station Name_BAYVIEW STATION,Station Name_BESSARION STATION,Station Name_BLOOR-YONGE STATION,Station Name_BROADVIEW STATION,Station Name_CASTLE FRANK STATION,Station Name_CHESTER STATION,...,Station Name_UNION STATION,Station Name_VAUGHAN METROPOLITAN CENTRE STATION,Station Name_VICTORIA PARK STATION,Station Name_WARDEN STATION,Station Name_WELLESLEY STATION,Station Name_WILSON STATION,Station Name_WOODBINE STATION,Station Name_YORK MILLS STATION,Station Name_YORK UNIVERSITY STATION,Station Name_YORKDALE STATION
0,2014-01-01 00:20:00,2,1.0,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,2014-01-01 02:00:00,2,1.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2014-01-01 02:40:00,2,1.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2014-01-01 03:10:00,2,1.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2014-01-01 03:20:00,2,1.0,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Note From Data: there is a maximum of 8 simultaneously being used

In [30]:
stations = lstm_data[lstm_data.columns[3:]]
stations = stations.astype(int)
display(stations)
print(stations.sum(axis=1).max())

,Station Name_BAY STATION,Station Name_BAYVIEW STATION,Station Name_BESSARION STATION,Station Name_BLOOR-YONGE STATION,Station Name_BROADVIEW STATION,Station Name_CASTLE FRANK STATION,Station Name_CHESTER STATION,Station Name_CHRISTIE STATION,Station Name_COLLEGE STATION,Station Name_COXWELL STATION,...,Station Name_UNION STATION,Station Name_VAUGHAN METROPOLITAN CENTRE STATION,Station Name_VICTORIA PARK STATION,Station Name_WARDEN STATION,Station Name_WELLESLEY STATION,Station Name_WILSON STATION,Station Name_WOODBINE STATION,Station Name_YORK MILLS STATION,Station Name_YORK UNIVERSITY STATION,Station Name_YORKDALE STATION
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277934,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
277935,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
277936,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
277937,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


8


In [31]:
columns_used = ['10_min_time', 'Day']
columns_used.extend(lstm_data.columns[3:])
lstm_data = lstm_data[columns_used]
lstm_data[lstm_data.columns[2:]]  = lstm_data[lstm_data.columns[2:]].astype(int)
lstm_data.to_csv(out_data_name, index=False)

In [32]:
display(lstm_data)

,10_min_time,Day,Station Name_BAY STATION,Station Name_BAYVIEW STATION,Station Name_BESSARION STATION,Station Name_BLOOR-YONGE STATION,Station Name_BROADVIEW STATION,Station Name_CASTLE FRANK STATION,Station Name_CHESTER STATION,Station Name_CHRISTIE STATION,...,Station Name_UNION STATION,Station Name_VAUGHAN METROPOLITAN CENTRE STATION,Station Name_VICTORIA PARK STATION,Station Name_WARDEN STATION,Station Name_WELLESLEY STATION,Station Name_WILSON STATION,Station Name_WOODBINE STATION,Station Name_YORK MILLS STATION,Station Name_YORK UNIVERSITY STATION,Station Name_YORKDALE STATION
0,2014-01-01 00:20:00,2,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,2014-01-01 02:00:00,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2014-01-01 02:40:00,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2014-01-01 03:10:00,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2014-01-01 03:20:00,2,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277934,2024-12-31 23:00:00,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
277935,2024-12-31 23:10:00,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
277936,2024-12-31 23:20:00,1,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
277937,2024-12-31 23:30:00,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Train/Test Splits

In [33]:
train_16_24 = lstm_data[lstm_data['10_min_time'] < '2024-01-01']
train_20_24 = train_16_24[train_16_24['10_min_time'] >= '2020-01-01']

test = lstm_data[lstm_data['10_min_time'] >= '2024-01-01']

train_16_24 = train_16_24.astype(int)
train_20_24 = train_20_24.astype(int)
test = test.astype(int)

In [34]:
def create_sliding_chunks(data, chunk_size, step):
    data = torch.tensor(data.values, dtype=torch.float32)
    chunks = []
    for i in range(0, len(data) - chunk_size + 1, step):
        chunks.append(data[i:i+chunk_size])
    return torch.stack(chunks)

In [39]:
seq_len = 2*2*24 #two days
input_size = 71
output_size = 69
hidden_size = 16
num_layers = 1
batch_size = 32

In [40]:
train_all = create_sliding_chunks(train_16_24, seq_len, 1)
train_recent = create_sliding_chunks(train_20_24, seq_len, 1)
test_seq = create_sliding_chunks(test, seq_len, 1)

## Data formatting

In [58]:
# these features are the output desired from the dataset (size (num_sequences, 69))
y_all = train_all[:, -1, 2:]

# make it so that for each sequence the last row has no info for the train delays (size (num_sequences, seq_len, 3))
X_all = train_all[:, :-1, :]

y_recent = train_recent[:, -1, 2:]

# do the same for recent data
X_recent = train_recent[:, :-1, :]

y_test = test_seq[:, -1, 2:]

# do the same for testing data
X_test = test_seq[:, :-1, :]

# Define LSTM Model

In [61]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        
        out = self.sigmoid(out) #probability of a stop

        return out

## Set up instances of LSTM Model

Instantiate model

In [62]:
model = LSTMModel(input_size, hidden_size, num_layers, output_size)
model_recent = LSTMModel(input_size, hidden_size, num_layers, output_size)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_recent.parameters(), lr=0.01)

# Train LSTM Model

In [63]:
num_epochs = 10

for epoch in range(num_epochs):
    for i in range(0, len(X_recent), batch_size):
        x_batch = X_recent[i:i+batch_size]
        y_batch = y_recent[i:i+batch_size]

        outputs = model_recent(x_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch: {epoch}, Cross Entropy: {loss}")

Epoch: 0, Cross Entropy: 0.6932068467140198
Epoch: 1, Cross Entropy: 0.693155825138092
Epoch: 2, Cross Entropy: 0.693148672580719
Epoch: 3, Cross Entropy: 0.6931475400924683
Epoch: 4, Cross Entropy: 0.6931471824645996
Epoch: 5, Cross Entropy: 0.6931471824645996


KeyboardInterrupt: 

# Test LSTM Model

In [65]:
model_recent.eval() 
with torch.no_grad():
    predictions = model_recent(X_test) 

mse = criterion(predictions, y_test)
print(f"Cross Entropy on test set: {mse.item()}")

Cross Entropy on test set: 0.6931474804878235


In [71]:
print(predictions)
print(predictions.mean(dim = 1).unique())

tensor([[2.1600e-08, 2.2959e-08, 2.3332e-08,  ..., 2.2414e-08, 1.9336e-08,
         2.2214e-08],
        [2.1600e-08, 2.2959e-08, 2.3332e-08,  ..., 2.2414e-08, 1.9336e-08,
         2.2214e-08],
        [2.1600e-08, 2.2959e-08, 2.3332e-08,  ..., 2.2414e-08, 1.9336e-08,
         2.2214e-08],
        ...,
        [2.1600e-08, 2.2959e-08, 2.3332e-08,  ..., 2.2414e-08, 1.9336e-08,
         2.2214e-08],
        [2.1600e-08, 2.2959e-08, 2.3332e-08,  ..., 2.2414e-08, 1.9336e-08,
         2.2214e-08],
        [2.1600e-08, 2.2959e-08, 2.3332e-08,  ..., 2.2414e-08, 1.9336e-08,
         2.2214e-08]])
tensor([2.2538e-08])


# Conclusions and Discussion

It is clear from the results of LSTM training and testing that the LSTM model on only the subway delays as a sequence that subsequent subway delays cannot be deduced from prior delays in a sequence. We note that tested predictions result in a standardized result. In other words, as most of the datapoints are of very low probability (no delay), the Binary Cross Entropy Loss with Logits is minimized by simply having a prediction which always concludes 0 (no delay).